<a href="https://colab.research.google.com/github/vitorleo/colab/blob/main/JoyVASA_Avatar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JoyVASA — Avatar Falante (Foto + Áudio MP3)

Este notebook usa o [JoyVASA](https://github.com/jdh-algo/JoyVASA) (extensão *audio-driven* do LivePortrait) para animar uma foto a partir de um áudio `.mp3` ou `.wav`.

**Como usar:**
1. Certifique-se de que o ambiente está usando **GPU (T4)** (`Ambiente de execução > Alterar tipo de ambiente de execução`).
2. Faça upload da sua **foto** (`.jpg`/`.png`) e do **áudio** (`.mp3`/`.wav`) na pasta `/content` (menu lateral esquerdo 📁).
3. Execute a **Célula 1** para instalar e baixar os modelos.
4. Execute a **Célula 2** para gerar o vídeo final em `/content/avatar_final.mp4`.

In [ ]:
#@title 1. Instalar JoyVASA e baixar modelos
import os
import shutil
from huggingface_hub import snapshot_download

!nvidia-smi

base_path = "/content"
%cd $base_path

if not os.path.exists(f"{base_path}/JoyVASA"):
    !git clone https://github.com/jdh-algo/JoyVASA.git

%cd $base_path/JoyVASA
!pip install -q uv huggingface_hub
!uv venv /content/py310 --python 3.10 --clear
!uv pip install --python /content/py310/bin/python -r requirements.txt
!uv pip install --python /content/py310/bin/python tyro huggingface_hub

weights_dir = f"{base_path}/JoyVASA/pretrained_weights"
os.makedirs(weights_dir, exist_ok=True)

print("Baixando LivePortrait...")
snapshot_download(
    repo_id="KwaiVGI/LivePortrait",
    local_dir=weights_dir,
    ignore_patterns=["*.git*", "README.md", "docs/*"]
)

print("Baixando JoyVASA...")
snapshot_download(
    repo_id="jdh-algo/JoyVASA",
    local_dir=f"{weights_dir}/JoyVASA",
    ignore_patterns=["*.git*", "README.md"]
)

print("Baixando chinese-hubert-base...")
snapshot_download(
    repo_id="TencentGameMate/chinese-hubert-base",
    local_dir=f"{weights_dir}/TencentGameMate:chinese-hubert-base",
    ignore_patterns=["*.git*", "README.md"]
)

print("Instalação e download concluídos!")

In [ ]:
#@title 2. Gerar vídeo do avatar a partir da foto e do MP3 em /content
import glob
import os
import shutil
import subprocess
from IPython.display import Video, display
from google.colab import files

#@markdown Deixe como `auto` para detectar automaticamente a foto e o áudio na pasta `/content`, ou informe o caminho completo:
imagem_path = "auto" #@param {type:"string"}
audio_path = "auto" #@param {type:"string"}
cfg_scale = 2.0 #@param {type:"number"}
baixar_automaticamente = True #@param {type:"boolean"}

def find_first_file(folder, extensions):
    candidates = []
    search_dirs = [folder, os.path.join(folder, "content")]
    for d in search_dirs:
        if not os.path.isdir(d):
            continue
        for ext in extensions:
            candidates.extend(glob.glob(os.path.join(d, f"*{ext}")))
            candidates.extend(glob.glob(os.path.join(d, f"*{ext.upper()}")))
    # Ignora arquivos gerados pelo próprio notebook
    candidates = [
        p for p in candidates
        if os.path.basename(p) not in {"input_audio.wav", "avatar_final.mp4"}
    ]
    candidates.sort(key=os.path.getmtime, reverse=True)
    return candidates[0] if candidates else None

if imagem_path.strip().lower() == "auto":
    imagem_path = find_first_file("/content", [".jpg", ".jpeg", ".png", ".webp"])
if audio_path.strip().lower() == "auto":
    audio_path = find_first_file("/content", [".mp3", ".wav", ".m4a", ".ogg"])

assert imagem_path and os.path.exists(imagem_path), "Nenhuma imagem encontrada em /content. Faça upload da foto!"
assert audio_path and os.path.exists(audio_path), "Nenhum áudio encontrado em /content. Faça upload do .mp3!"

print(f"Foto selecionada : {imagem_path}")
print(f"Áudio selecionado: {audio_path}")

# Converte o áudio para WAV 16kHz mono para evitar problemas de compatibilidade
wav_path = "/content/input_audio.wav"
subprocess.run(
    ["ffmpeg", "-y", "-i", audio_path, "-ar", "16000", "-ac", "1", wav_path],
    check=True,
    capture_output=True
)

%cd /content/JoyVASA
before_mp4s = set(glob.glob("/content/JoyVASA/**/*.mp4", recursive=True))

!/content/py310/bin/python inference.py -r "$imagem_path" -a "$wav_path" --animation_mode human --cfg_scale $cfg_scale

after_mp4s = set(glob.glob("/content/JoyVASA/**/*.mp4", recursive=True))
new_mp4s = sorted(after_mp4s - before_mp4s, key=os.path.getmtime, reverse=True)

if not new_mp4s:
    raise RuntimeError("Nenhum vídeo .mp4 novo foi gerado. Verifique os logs acima.")

# Prefere o vídeo sem '_concat' se houver mais de um
non_concat = [p for p in new_mp4s if "concat" not in os.path.basename(p).lower()]
generated_video = non_concat[0] if non_concat else new_mp4s[0]

final_output = "/content/avatar_final.mp4"
shutil.copy2(generated_video, final_output)
print(f"\nVídeo final salvo em: {final_output}")

display(Video(final_output, embed=True, width=512))

if baixar_automaticamente:
    files.download(final_output)